# Modular Exponentiation

## Shor's algorithm needs to compute $a^x \bmod N$ for a superposition of $x$. Equivalently, it needs to implement the unitary $U_{a, N} : |x\rangle|y\rangle \mapsto |x\rangle|y \cdot a^x \bmod N\rangle$. Doing this efficiently is the *classical* engineering at the heart of Shor — the QFT is asymptotically free by comparison.

# 1. The operation we need

## For phase estimation on the modular multiplier $M_a : |y\rangle \mapsto |a y \bmod N\rangle$, we need controlled $M_a^{2^k} = M_{a^{2^k} \bmod N}$ for $k = 0, 1, \dots, n-1$. Each is a modular multiplier with a different base, which we can precompute classically:

$$ \Large  a_0 = a, \quad a_{k+1} = a_k^2 \bmod N. $$

## So the **quantum** part is just $n$ controlled modular multipliers, each at a different precomputed base.

# 2. Building a controlled modular multiplier

## $M_a : |y\rangle \mapsto |a y \bmod N\rangle$ is unitary because $a$ is invertible mod $N$ (we have already ensured $\gcd(a, N) = 1$). It can be built from:

## - **Quantum modular adders** (each implements $|y\rangle \mapsto |y + c \bmod N\rangle$ for classical $c$).
## - A control bit so that $|0\rangle|y\rangle \mapsto |0\rangle|y\rangle$ and $|1\rangle|y\rangle \mapsto |1\rangle|a y \bmod N\rangle$.
## - Standard arithmetic constructions (carry-save, Draper's QFT-based adder, etc.).

## All in all, the cost is $O(n^3)$ gates, which is polynomial in $\log N$ — and that's enough.

In [ ]:
# We won't build a full quantum modular multiplier here, but let's at least
# precompute the bases a_k = a^{2^k} mod N classically.
def precompute_bases(a, N, n):
    bases = []
    a_k = a % N
    for _ in range(n):
        bases.append(a_k)
        a_k = (a_k * a_k) % N
    return bases

a, N, n = 7, 15, 8
bases = precompute_bases(a, N, n)
print(f'a={a}, N={N}')
for k, b in enumerate(bases):
    print(f'  a^(2^{k}) mod N = {b}')

# 3. Why this matters for resource estimates

## When people quote "Shor's algorithm requires $\sim 2n^3$ Toffolis" or estimate $\sim 20$ million physical qubits to break RSA-2048, the dominant cost is **modular exponentiation**, not the QFT. Most of Shor's research literature in the last decade has focused on shrinking this number with cleverer arithmetic circuits.

# Recap

## - Shor needs controlled $|y\rangle \mapsto |a^{2^k} y \bmod N\rangle$ for $k = 0, \dots, n-1$.
## - The bases $a^{2^k} \bmod N$ are precomputed classically — quick.
## - The quantum cost is $O(n^3)$ gates, polynomial in $\log N$.
## - This is the practical bottleneck of fault-tolerant Shor.

## Next: putting QPE + modular exponentiation together to do **order finding**.